In [1]:
import torch
print(torch.__version__)
import sys
print(sys.executable)

2.10.0+cpu
e:\FineTunning\ft\python.exe


In [2]:
import os
print(os.environ.get("SSL_CERT_FILE"))

E:\FineTunning\ft\Library\ssl\cacert.pem


In [3]:
import os
os.environ.pop("SSL_CERT_FILE", None)

'E:\\FineTunning\\ft\\Library\\ssl\\cacert.pem'

In [4]:
import os
import certifi

os.environ["SSL_CERT_FILE"] = certifi.where()

In [5]:
from datasets import load_dataset
from transformers import AutoTokenizer
from transformers import AutoModelForCausalLM

e:\FineTunning\ft\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
dataset = load_dataset("json",data_files="dataset.json")
dataset

DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output'],
        num_rows: 5
    })
})

In [7]:
print(dataset["train"][0])

{'instruction': 'Translate English to French', 'input': 'Hello', 'output': 'Bonjour'}


In [8]:
def format_prompt(example):
    prompt = f"""
###Instruction:
{example["instruction"]}
###Input:
{example["input"]}
###Response:
{example["output"]}
"""
    return {"text" : prompt}

In [9]:
dataset  = dataset.map(format_prompt)

In [10]:
dataset

DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output', 'text'],
        num_rows: 5
    })
})

In [11]:
dataset["train"][0]

{'instruction': 'Translate English to French',
 'input': 'Hello',
 'output': 'Bonjour',
 'text': '\n###Instruction:\nTranslate English to French\n###Input:\nHello\n###Response:\nBonjour\n'}

In [12]:
tokenizer = AutoTokenizer.from_pretrained("distilgpt2")


In [13]:
tokenizer.pad_token = tokenizer.eos_token

In [31]:
def tokenize_function(example):
    tokens = tokenizer(
        example["text"],
        truncation= True,
        padding = "max_length",
        max_length = 128 
    )
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

In [32]:
tokenized_dataset = dataset.map(tokenize_function)

Map: 100%|██████████| 5/5 [00:00<00:00, 55.68 examples/s]


In [33]:
tokenized_dataset["train"]

Dataset({
    features: ['instruction', 'input', 'output', 'text', 'input_ids', 'attention_mask', 'labels'],
    num_rows: 5
})

In [34]:

from peft import LoraConfig, get_peft_model

In [35]:
model = AutoModelForCausalLM.from_pretrained("distilgpt2")

Loading weights: 100%|██████████| 76/76 [00:00<00:00, 11425.75it/s]


In [36]:
lora_config = LoraConfig(
    r=8,
    lora_alpha = 16,
    target_modules = ["c_attn"],
    lora_dropout=0.1,
    bias = "none",
    task_type = "CAUSAL_LM"
)

In [37]:
model = get_peft_model(model,lora_config)
model.print_trainable_parameters()

trainable params: 147,456 || all params: 82,060,032 || trainable%: 0.1797


e:\FineTunning\ft\Lib\site-packages\peft\tuners\lora\layer.py:2285: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(


In [38]:
from transformers import TrainingArguments, Trainer

In [42]:
training_args = TrainingArguments(

    output_dir = "./results",
        per_device_train_batch_size=2,
    
    num_train_epochs=3,
    
    logging_steps=1,
    
    save_steps=50,
    
    learning_rate=2e-4,
    
    fp16=True


)

In [43]:
trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset = tokenized_dataset["train"]
)

In [44]:
trainer.train()

e:\FineTunning\ft\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
1,8.836362
2,9.461185
3,9.023534
4,8.967044
5,9.071161
6,8.937100
7,9.082360
8,9.114500
9,8.825287


TrainOutput(global_step=9, training_loss=9.035392655266655, metrics={'train_runtime': 3.9478, 'train_samples_per_second': 3.8, 'train_steps_per_second': 2.28, 'total_flos': 491630100480.0, 'train_loss': 9.035392655266655, 'epoch': 3.0})

In [45]:
model.save_pretrained("FineTunedModel")

In [46]:
input_text = """
### Instruction:
Translate English to French

### Input:
Good night

### Response:
"""

In [48]:
inputs = tokenizer(input_text, return_tensors="pt")


In [49]:
output = model.generate(**inputs, max_length=100)
print(tokenizer.decode(output[0]))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.



### Instruction:
Translate English to French

### Input:
Good night

### Response:
Good night
###
###
###
###
###
###
###
###
###
###
###
###
###
###
###
###
###
###
###
###
###
###
###
###
###
###
###
###
###
###
###
###
###
###
###
###
###
